In [18]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.transforms import v2

from torchvision.models import swin_t, Swin_T_Weights, resnet50

# ==========================================
# 1. КОНФИГУРАЦИЯ И КОНСТАНТЫ
# ==========================================

DATA_ROOT = './data'
SUBMISSION_NAME = "submission_blend_v2"

CONFIG = {
    "image_size": (256, 512),  # (H, W)
    "bev_resolution": 0.8,     # метров на пиксель
    "bev_grid_shape": (188, 126), # (H_bev, W_bev) -> (Forward, Side)
    # Границы сетки относительно машины (в метрах)
    # Машина смотрит вперед (ось X в системе машины, но H в тензоре карты)
    # Центр сзади машины. 
    # Forward: 0 .. 188*0.8 = 150.4м
    # Side: -63*0.8 .. +63*0.8 = -50.4 .. 50.4м
    "x_bound": (0, 150.4),    # Вперед
    "y_bound": (-50.4, 50.4), # Влево/Вправо
    "z_bound": (-10, 10),     # Высота (для фильтрации точек)
    "d_bound": (4.0, 100.0, 1.0), # Глубина: мин, макс, шаг
    
    "batch_size": 16,
    "num_workers": 4,
    "lr": 3e-4,
    "epochs": 10,
    "device": "cuda:1" # if torch.cuda.is_available() else "cpu"
}

CAMERA_NAMES = [
    "/camera/inner/frontal/middle",
    "/camera/inner/frontal/far",
    "/side/left/forward",
    "/side/right/forward",
]
# Соответствующие пути к матрицам
INTRINSICS_NAMES = [x + "/intrinsic_params" for x in CAMERA_NAMES]
CAR2CAM_NAMES = [x + "/car_to_cam" for x in CAMERA_NAMES]

In [2]:
class BEVDataset(Dataset):
    def __init__(self, data_dir: Path, mode: str = "train", config=CONFIG):
        """
        data_dir: Путь к папке конкретного сплита (например, .../dataset_train)
        mode: 'train', 'val' (возвращают GT) или 'test' (не возвращает GT)
        """
        self.mode = mode
        self.data_dir = Path(data_dir)
        self.config = config
        
        # Читаем CSV. Ожидается, что info.csv лежит внутри data_dir
        csv_path = self.data_dir / "info.csv"
        if not csv_path.exists():
            raise FileNotFoundError(f"CSV file not found at {csv_path}")
            
        self.info = pd.read_csv(csv_path, index_col=0)
        
        # Корневая папка для путей из CSV.
        # Если в CSV пути вида "autonomy_yandex_dataset_train/images/...", 
        # а data_dir = ".../autonomy_yandex_dataset_train", 
        # то нам нужно подняться на уровень выше.
        self.data_root = self.data_dir.parent 
        
        # Препроцессинг
        self.transform = v2.Compose([
            v2.ToImage(), 
            v2.ToDtype(torch.float32, scale=True),
            v2.Resize(config["image_size"]),
            v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        ])
        
        # Кэшируем список семплов для быстрого доступа по индексу
        self.samples = []
        for idx, row in self.info.iterrows():
            sample = {
                "images": [row[name] for name in CAMERA_NAMES],
                "intrinsics": [row[name] for name in INTRINSICS_NAMES],
                "car2cam": [row[name] for name in CAR2CAM_NAMES],
            }
            # GT нужен только для train и val
            if self.mode in ["train", "val"]:
                sample["gt_grid"] = row["gt_occupancy_grid"]
            self.samples.append(sample)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        imgs = []
        intrinsics = []
        car2cams = []
        
        for i in range(4):
            # 1. Загрузка изображения
            # Используем data_root, так как пути в CSV относительные
            img_path = self.data_root / sample["images"][i]
            pil_img = Image.open(img_path)
            orig_w, orig_h = pil_img.size
            
            # 2. Трансформация
            img_tensor = self.transform(pil_img)
            imgs.append(img_tensor)
            
            # 3. Загрузка Intrinsics
            intr_path = self.data_root / sample["intrinsics"][i]
            intr = np.load(intr_path) 
            
            # --- Fix 3x4 to 3x3 ---
            if intr.shape == (3, 4):
                intr = intr[:3, :3]
            
            # Масштабирование матрицы камеры под ресайз картинки
            scale_x = self.config["image_size"][1] / orig_w
            scale_y = self.config["image_size"][0] / orig_h
            intr[0, 0] *= scale_x # fx
            intr[0, 2] *= scale_x # cx
            intr[1, 1] *= scale_y # fy
            intr[1, 2] *= scale_y # cy
            
            intrinsics.append(torch.from_numpy(intr).float())
            
            # 4. Загрузка Extrinsics
            c2c_path = self.data_root / sample["car2cam"][i]
            c2c = np.load(c2c_path)
            car2cams.append(torch.from_numpy(c2c).float())

        imgs = torch.stack(imgs)
        intrinsics = torch.stack(intrinsics)
        car2cams = torch.stack(car2cams)
        
        # Инверсия: нам нужно camera -> car (для LSS)
        cam2cars = torch.inverse(car2cams)

        # Возврат данных в зависимости от режима
        if self.mode in ["train", "val"]:
            gt_path = self.data_root / sample["gt_grid"]
            gt_grid = np.load(gt_path).astype(np.int64)
            if len(gt_grid.shape) == 2:
                gt_grid = gt_grid[None, ...] # (1, H, W)
            return imgs, intrinsics, cam2cars, torch.from_numpy(gt_grid)
        
        # Для теста GT не возвращаем
        return imgs, intrinsics, cam2cars

In [3]:
from torchvision.models import swin_t, Swin_T_Weights


class CamEncoder(nn.Module):
    """
    Извлекает признаки и предсказывает распределение глубины для изображения.
    Backbone: Swin Transformer (swin_t).
    """
    def __init__(self, D, C, downsample=32):
        super(CamEncoder, self).__init__()
        self.D = D  # depth bins
        self.C = C  # context channels

        # --- SWIN BACKBONE ---
        # swin_t: patch_size=4, итоговый stride = 32 (как у ResNet50 layer4)
        weights = Swin_T_Weights.IMAGENET1K_V1
        self.backbone = swin_t(weights=weights)

        # Кол-во каналов последнего уровня (embed_dim * 2 ** (len(depths)-1))
        # Это то же число, что и in_features у классификационной головы.
        in_channels = self.backbone.head.in_features  # для swin_t = 768

        # --- HEAD ДЛЯ ГЛУБИНЫ+КОНТЕКСТА ---
        # Вход: (B, in_channels, H/32, W/32)
        # Выход: (B, D + C, H/32, W/32)
        self.depth_net = nn.Conv2d(in_channels, self.D + self.C, kernel_size=1)

    def get_depth_dist(self, x):
        # x: (B, D + C, H, W)
        return x[:, :self.D].softmax(dim=1)

    def get_context(self, x):
        return x[:, self.D:]

    def forward(self, x):
        # x: (B*N, 3, H, W)

        # Swin сам делает patch embedding и все стадии ↓ разрешения.
        # features: (B*N, H/32, W/32, C)  — ВАЖНО: channel last!
        x = self.backbone.features(x)          # (B, H', W', C)
        x = self.backbone.norm(x)              # (B, H', W', C)

        # Переводим в BCHW (channel first)
        # Можно использовать встроенный permute-модуль из модели:
        x = self.backbone.permute(x)           # (B, C, H', W')

        # Дальше всё как раньше
        x = self.depth_net(x)                  # (B, D + C, H', W')
        depth = self.get_depth_dist(x)         # (B, D, H', W')
        context = self.get_context(x)          # (B, C, H', W')

        return depth, context

In [4]:
class ViewTransformer(nn.Module):
    """
    Переводит 2D признаки в 3D облако точек и проецирует их в BEV сетку.
    """
    def __init__(self, grid_conf, data_conf):
        super(ViewTransformer, self).__init__()
        self.grid_conf = grid_conf
        self.data_conf = data_conf
        
        dx, bx, nx = self.gen_dx_bx(self.grid_conf['x_bound'], self.grid_conf['y_bound'], self.grid_conf['z_bound'])
        self.dx = nn.Parameter(dx, requires_grad=False)
        self.bx = nn.Parameter(bx, requires_grad=False)
        self.nx = nn.Parameter(nx, requires_grad=False)

        self.D = int((data_conf['d_bound'][1] - data_conf['d_bound'][0]) / data_conf['d_bound'][2])
        self.frustum = self.create_frustum()

    def gen_dx_bx(self, xbound, ybound, zbound):
        dx = torch.Tensor([0.8, 0.8, 20.0]) 
        bx = torch.Tensor([xbound[0] + 0.4, ybound[0] + 0.4, zbound[0]])
        nx = torch.LongTensor([(xbound[1] - xbound[0]) / 0.8, (ybound[1] - ybound[0]) / 0.8, 1])
        return dx, bx, nx

    def create_frustum(self):
        feat_h, feat_w = 8, 16 
        ds = torch.arange(*self.data_conf['d_bound'], dtype=torch.float).view(-1, 1, 1).expand(-1, feat_h, feat_w)
        D, H, W = ds.shape
        xs = torch.linspace(0, self.data_conf['image_size'][1] - 1, W).view(1, 1, W).expand(D, H, W)
        ys = torch.linspace(0, self.data_conf['image_size'][0] - 1, H).view(1, H, 1).expand(D, H, W)
        frustum = torch.stack((xs, ys, ds), -1)
        return nn.Parameter(frustum, requires_grad=False)

    def get_geometry(self, rots, trans, intrinsics):
        B, N, _ = trans.shape
        points = self.frustum.unsqueeze(0).unsqueeze(0).unsqueeze(-1)
        points = torch.cat((points[:, :, :, :, :, :2] * points[:, :, :, :, :, 2:3], points[:, :, :, :, :, 2:3]), 5)
        combined_transform = torch.inverse(intrinsics)
        points = combined_transform.view(B, N, 1, 1, 1, 3, 3).matmul(points).squeeze(-1)
        points = rots.view(B, N, 1, 1, 1, 3, 3).matmul(points.unsqueeze(-1)).squeeze(-1)
        points += trans.view(B, N, 1, 1, 1, 3)
        return points

    def voxel_pooling(self, geom_feats, x):
        """
        Складывает фичи, попавшие в один воксель.
        """
        B, N, D, H, W, _ = geom_feats.shape
        
        # --- ИСПРАВЛЕНИЕ ЗДЕСЬ ---
        # Используем reshape вместо view, так как данные могут быть не contiguous
        # Либо явно вызываем contiguous()
        geom_feats = geom_feats.reshape(B, -1, 3) 
        feats = x.reshape(B, -1, x.shape[-1])
        # -------------------------
        
        long_coords = ((geom_feats - self.bx.to(geom_feats.device)) / self.dx.to(geom_feats.device)).long()
        
        valid = (long_coords[:, :, 0] >= 0) & (long_coords[:, :, 0] < self.nx[0]) & \
                (long_coords[:, :, 1] >= 0) & (long_coords[:, :, 1] < self.nx[1]) & \
                (long_coords[:, :, 2] >= 0) & (long_coords[:, :, 2] < self.nx[2])
        
        bev_map = torch.zeros((B, self.data_conf['bev_grid_shape'][0], self.data_conf['bev_grid_shape'][1], feats.shape[-1]), device=feats.device)
        
        for b in range(B):
            cur_valid = valid[b]
            cur_coords = long_coords[b][cur_valid]
            cur_feats = feats[b][cur_valid]
            
            if cur_coords.shape[0] == 0:
                continue
            
            bev_map[b].index_put_((cur_coords[:, 0], cur_coords[:, 1]), cur_feats, accumulate=True)

        return bev_map.permute(0, 3, 1, 2).contiguous()

    def forward(self, x, rots, trans, intrinsics):
        B, N, C, D, H, W = x.shape
        x = x.permute(0, 1, 3, 4, 5, 2) # (B, N, D, H, W, C)
        geom = self.get_geometry(rots, trans, intrinsics) 
        bev = self.voxel_pooling(geom, x)
        return bev

class BEVModel(nn.Module):
    def __init__(self, conf):
        super(BEVModel, self).__init__()
        self.conf = conf
        self.C = 64 # Каналы признаков
        self.D = int((conf['d_bound'][1] - conf['d_bound'][0]) / conf['d_bound'][2])
        
        self.cam_encoder = CamEncoder(self.D, self.C, downsample=32)
        self.view_transformer = ViewTransformer(conf, conf)
        
        # Декодер BEV (простой U-Net like или ResNet блоки)
        # Вход: 64 канала. Выход: 1 канал (logits)
        self.bev_decoder = nn.Sequential(
            nn.Conv2d(self.C, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 1, 1) # Итоговый предиктор
        )

    def forward(self, images, intrinsics, cam2cars):
        # images: (B, 4, 3, H, W)
        B, N, C, H, W = images.shape
        
        # 1. CamEncoder
        # Сливаем батч и камеры: (B*N, 3, H, W)
        images_flat = images.view(B*N, C, H, W)
        depth_dist, context = self.cam_encoder(images_flat)
        
        # Outer product: (B*N, C, D, H_f, W_f)
        # context: (B*N, C, 8, 16), depth: (B*N, D, 8, 16)
        x = context.unsqueeze(2) * depth_dist.unsqueeze(1)
        
        # Reshape обратно в батч
        # (B, N, C, D, H_f, W_f)
        x = x.view(B, N, self.C, self.D, x.shape[-2], x.shape[-1])
        
        # 2. View Transformer (Lift & Splat)
        # Разбираем cam2cars на rotation и translation
        rots = cam2cars[:, :, :3, :3]
        trans = cam2cars[:, :, :3, 3]
        
        bev_feat = self.view_transformer(x, rots, trans, intrinsics)
        
        # 3. BEV Decoder (Shoot)
        logits = self.bev_decoder(bev_feat) # (B, 1, 188, 126)
        
        return logits

In [6]:
def calculate_iou_new(preds, targets, ignore_index=255):
    """
    Считает Macro IoU: (IoU_class_0 + IoU_class_1) / 2.
    Игнорирует пиксели со значением ignore_index (обычно 255).
    """
    # 1. Получаем бинарные предсказания (0 или 1)
    preds_sigmoid = torch.sigmoid(preds)
    preds_bin = preds_sigmoid > 0.5  # True, где предсказан класс 1
    
    # 2. Создаем маску валидных пикселей (исключаем 255)
    valid_mask = (targets != ignore_index)
    
    # --- IoU для Класса 1 (Занято / Occupied) ---
    # Intersection: Предсказано 1 И Истина 1 (в валидной зоне)
    tp_1 = (preds_bin & (targets == 1) & valid_mask).sum().float()
    # Union: Предсказано 1 ИЛИ Истина 1 (в валидной зоне)
    union_1 = ((preds_bin | (targets == 1)) & valid_mask).sum().float()
    
    iou_1 = (tp_1 + 1e-6) / (union_1 + 1e-6)
    
    # --- IoU для Класса 0 (Свободно / Free) ---
    # Intersection: Предсказано 0 И Истина 0 (в валидной зоне)
    # ~preds_bin означает "НЕ 1", то есть 0
    tp_0 = ((~preds_bin) & (targets == 0) & valid_mask).sum().float()
    # Union: Предсказано 0 ИЛИ Истина 0 (в валидной зоне)
    union_0 = (((~preds_bin) | (targets == 0)) & valid_mask).sum().float()
    
    iou_0 = (tp_0 + 1e-6) / (union_0 + 1e-6)
    
    # --- Macro Average ---
    macro_iou = (iou_1 + iou_0) / 2
    
    return macro_iou.item()

In [ ]:
model_danil_1 = BEVModel(CONFIG).to(CONFIG['device'])
model_danil_1.load_state_dict(torch.load("./checkpoints/lss_model_epoch_9.pth"))

<All keys matched successfully>

In [13]:
def generate_submission_danil(model):
    test_dir = Path(DATA_ROOT, "autonomy_yandex_dataset_test") 

    test_dataset = BEVDataset(test_dir,   mode="test",   config=CONFIG)
    test_loader = DataLoader(
        test_dataset, 
        batch_size=1,
        shuffle=False,            
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )
    test_info = pd.read_csv(Path(DATA_ROOT, "autonomy_yandex_dataset_test/info.csv") , index_col=0)
    model.eval()
    
    SUB_DIR = Path(f"{SUBMISSION_NAME}/predicted_static_grids")
    SUB_DIR.mkdir(parents=True, exist_ok=True)
    
    with torch.no_grad():
        for i, (images, intrinsics, car2cams) in enumerate(tqdm(test_loader, desc="Inference")):
            images, intrinsics, car2cams = images.to(CONFIG['device']), intrinsics.to(CONFIG['device']), car2cams.to(CONFIG['device'])
            
            preds = model.forward(images, intrinsics, car2cams)
            probs = torch.sigmoid(preds).cpu().numpy()

            # Get path from info
            orig_rel_path = test_info.iloc[i]["predicted_occupancy_grid"]
            filename = Path(orig_rel_path).name
            save_path = SUB_DIR / filename
            
            # Если файл уже существует — загружаем и добавляем
            if save_path.exists():
                try:
                    existing_grid = np.load(save_path)
                    probs = existing_grid + probs
                except Exception as e:
                    print(f"Ошибка при загрузке {save_path}: {e}")

            # Сохраняем результат (либо как новый файл, либо обновленный)
            np.save(save_path, probs)
            
    print("Inference complete. Zip the 'submission' folder.")

generate_submission_danil(model_danil_1)

Inference: 100%|██████████| 2000/2000 [00:48<00:00, 41.48it/s]

Inference complete. Zip the 'submission' folder.


In [14]:
import math
import torch


def positionalencoding1d(d_model, length):
    """
    :param d_model: dimension of the model
    :param length: length of positions
    :return: length*d_model position matrix
    """
    if d_model % 2 != 0:
        raise ValueError("Cannot use sin/cos positional encoding with "
                         "odd dim (got dim={:d})".format(d_model))
    pe = torch.zeros(length, d_model)
    position = torch.arange(0, length).unsqueeze(1)
    div_term = torch.exp((torch.arange(0, d_model, 2, dtype=torch.float) *
                         -(math.log(10000.0) / d_model)))
    pe[:, 0::2] = torch.sin(position.float() * div_term)
    pe[:, 1::2] = torch.cos(position.float() * div_term)

    return pe


def positionalencoding2d(height, width, d_model):
    """
    :param d_model: dimension of the model
    :param height: height of the positions
    :param width: width of the positions
    :return: d_model*height*width position matrix
    """
    if d_model % 4 != 0:
        raise ValueError("Cannot use sin/cos positional encoding with "
                         "odd dimension (got dim={:d})".format(d_model))
    pe = torch.zeros(d_model, height, width)
    # Each dimension use half of d_model
    d_model = int(d_model / 2)
    div_term = torch.exp(torch.arange(0., d_model, 2) *
                         -(math.log(10000.0) / d_model))
    pos_w = torch.arange(0., width).unsqueeze(1)
    pos_h = torch.arange(0., height).unsqueeze(1)
    pe[0:d_model:2, :, :] = torch.sin(pos_w * div_term).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
    pe[1:d_model:2, :, :] = torch.cos(pos_w * div_term).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
    pe[d_model::2, :, :] = torch.sin(pos_h * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)
    pe[d_model + 1::2, :, :] = torch.cos(pos_h * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)

    return pe.permute(1, 2, 0)

In [ ]:
class MultiViewBEVModelResNet(nn.Module):
    def __init__(self, bev_h=48, bev_w=32, num_classes=1, num_cameras=4):
        super().__init__()
        
        # --- Backbone ---
        resnet = resnet50(weights='DEFAULT')
        self.encoder = nn.Sequential(*list(resnet.children())[:-2])
        # for p in self.encoder.parameters():
        #     p.requires_grad = False
        
        self.neck = nn.Conv2d(2048, 512, kernel_size=1)

        # --- BEV queries ---
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.num_classes = num_classes
        self.num_cameras = num_cameras
        
        self.bev_queries = nn.Parameter(torch.empty(bev_h * bev_w, 512))
        nn.init.xavier_normal_(self.bev_queries)
        
        # --- Transformer ---
        self.transformer = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(512, 8, batch_first=True),
            num_layers=4,
        )
        
        # --- Positional encoding (learnable) ---
        # self.pos_encoding = nn.Parameter(torch.empty(1, num_cameras * 8 * 16, 512))  # H_feat=8, W_feat=16
        # nn.init.xavier_normal_(self.pos_encoding)

        self.register_buffer("pos_embed", positionalencoding2d(8, 16, 512))  # H_feat=8, W_feat=16
        self.register_buffer("bev_pos_embed", positionalencoding2d(self.bev_h, self.bev_w, 512) )

        self.upsample = nn.Sequential(
            # Stage 1: coarse → medium (~48x32)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(size=(96,64), mode='bilinear', align_corners=False),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(),

            # Stage 2: medium → finer (~94x63)
            nn.BatchNorm2d(256),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(size=(188,126), mode='bilinear', align_corners=False),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(),

            # Stage 3: finer → final (188x126)
            nn.BatchNorm2d(128),
            # nn.Upsample(size=(188,126), mode='bilinear', align_corners=False),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            # Final projection
            nn.BatchNorm2d(64),
            nn.Conv2d(64, num_classes, kernel_size=1)
        )

        self.intrinsics_encoder = nn.Sequential(
            nn.BatchNorm1d(12),
            nn.Linear(12, 512),
            nn.ReLU(),
            nn.Linear(512, 512)
        )

        self.car2cams_encoder = nn.Sequential(
            nn.BatchNorm1d(16),
            nn.Linear(16, 512),
            nn.ReLU(),
            nn.Linear(512, 512)
        )


    def forward(self, x, intrinsics, car2cams):
        B, N, C, H, W = x.shape
        assert N == self.num_cameras, "Number of cameras mismatch"
        
        intrinsics = self.intrinsics_encoder(intrinsics.reshape(B * N, -1).float()).reshape(B, N, -1)
        car2cams = self.car2cams_encoder(car2cams.reshape(B * N, -1).float()).reshape(B, N, -1)

        # --- Extract features from each camera ---
        x = x.view(B * N, C, H, W)
        features = self.encoder(x)  # (B*N, 512, 8,16)
        features = self.neck(features)
        _, C_feat, H_feat, W_feat = features.shape
        features = features.view(B, N, C_feat, H_feat, W_feat)
        
        # --- Flatten camera + spatial dims for transformer ---
        features = features.permute(0, 1, 3, 4, 2)  # (B, N, H_feat, W_feat, C)
        features = features + self.pos_embed[None, None]  # (B, N, H_feat, W_feat, C)
        features = features.reshape(B, N, H_feat*W_feat, C_feat)  # (B, N, H_feat*W_feat, 512)
        features = features + intrinsics.unsqueeze(2) + car2cams.unsqueeze(2) 
        features = features.reshape(B, N*H_feat*W_feat, C_feat)  # (B, seq_len, 512)

        # --- Add positional encoding ---
        # features = features + self.pos_encoding  # (B, seq_len, 512)
        
        bev_tokens = self.bev_queries.reshape(self.bev_h, self.bev_w, C_feat) + self.bev_pos_embed
        bev_tokens = bev_tokens.view(self.bev_h*self.bev_w, C_feat)
        # --- Transformer decoder ---
        bev_tokens = self.transformer(
            self.bev_queries.unsqueeze(0).repeat(B,1,1),  # (B, bev_h*bev_w, 512)
            features
        )  # (B, bev_h*bev_w, 512)
        
        # --- Reshape to coarse BEV map ---
        bev_tokens = bev_tokens.transpose(1, 2).view(B, 512, self.bev_h, self.bev_w)  # (B, 512, H_bev, W_bev)
        
        # --- Conv upsampling to final BEV ---
        bev_logits = self.upsample(bev_tokens)  # (B, num_classes, 188, 126)
        
        return bev_logits

In [ ]:
model_sasha_1 = MultiViewBEVModelResNet()
model_sasha_1.load_state_dict(torch.load('./checkpoints/17_0.5551023058891297.pth'))
# model_sasha.load_state_dict(torch.load('/app/checkpoints/9_0.5645387780666351.pth'))
model_sasha_1 = model_sasha_1.to(CONFIG['device'])

In [19]:
CAMERA_NAMES = [
    "/camera/inner/frontal/middle",
    "/camera/inner/frontal/far",
    "/side/left/forward",
    "/side/right/forward",
]

INTRINSICS_NAMES = [
    "/camera/inner/frontal/middle/intrinsic_params",
    "/camera/inner/frontal/far/intrinsic_params",
    "/side/left/forward/intrinsic_params",
    "/side/right/forward/intrinsic_params",
]

CAR2CAM_NAMES = [
    "/camera/inner/frontal/middle/car_to_cam",
    "/camera/inner/frontal/far/car_to_cam",
    "/side/left/forward/car_to_cam",
    "/side/right/forward/car_to_cam",
]

GRIDS_NAMES = [
    "gt_occupancy_grid",
]

class BaseDataset(Dataset):
    def __init__(self, data_dir: Path, mode: str = "train"):
        self.mode = mode
        self.transform = v2.Compose([
            v2.PILToTensor(),
            v2.Resize((256, 512)),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        ])
        self.data_dir = data_dir
        self.info = pd.read_csv(data_dir / "info.csv", index_col=0)
        self.images_paths = []
        self.intrinsics_paths = []
        self.car2cam_paths = []
        if self.mode != "test":
            self.static_grids_paths = []

        for _, row in self.info.iterrows():
            self.images_paths.append([row[name] for name in CAMERA_NAMES])                
            self.intrinsics_paths.append([row[name] for name in INTRINSICS_NAMES])
            self.car2cam_paths.append([row[name] for name in CAR2CAM_NAMES])
            if self.mode != "test":
                self.static_grids_paths.append([row[name] for name in GRIDS_NAMES])


    def __len__(self):
        return len(self.info)

    def __getitem__(self, idx):
        images = torch.stack([self.transform(Image.open(self.data_dir.parent / img_path)) for img_path in self.images_paths[idx]], dim=0)
        intrinsics = np.stack([np.load(self.data_dir.parent / intr_path) for intr_path in self.intrinsics_paths[idx]], axis=0)
        car2cams = np.stack([np.load(self.data_dir.parent / car2cam_path) for car2cam_path in self.car2cam_paths[idx]], axis=0)

        if self.mode != "test":
            static_grids = np.load(self.data_dir.parent / self.static_grids_paths[idx][0]) 
            return images, intrinsics, car2cams, static_grids

        return images, intrinsics, car2cams

In [21]:
def generate_submission_sasha(model):
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    test_ds = BaseDataset(data_dir=Path(DATA_ROOT, "autonomy_yandex_dataset_test") , mode="test")
    test_loader = DataLoader(test_ds, batch_size=1, num_workers=8)
    
    test_info = pd.read_csv(Path(DATA_ROOT, "autonomy_yandex_dataset_test/info.csv") , index_col=0)
    model.eval()
    
    SUB_DIR = Path(f"{SUBMISSION_NAME}/predicted_static_grids")
    SUB_DIR.mkdir(parents=True, exist_ok=True)
    
    with torch.no_grad():
        for i, (images, intrinsics, car2cams) in enumerate(tqdm(test_loader, desc="Inference")):
            images, intrinsics, car2cams = images.to(CONFIG['device']), intrinsics.to(CONFIG['device']), car2cams.to(CONFIG['device'])
            
            preds = model.forward(images, intrinsics, car2cams)
            probs = torch.sigmoid(preds).cpu().numpy()

            # Get path from info
            orig_rel_path = test_info.iloc[i]["predicted_occupancy_grid"]
            filename = Path(orig_rel_path).name
            save_path = SUB_DIR / filename
            
            # Если файл уже существует — загружаем и добавляем
            if save_path.exists():
                try:
                    existing_grid = np.load(save_path)
                    probs = existing_grid + probs
                except Exception as e:
                    print(f"Ошибка при загрузке {save_path}: {e}")

            # Сохраняем результат (либо как новый файл, либо обновленный)
            np.save(save_path, probs)
            
    print(f"Inference complete. Zip the {SUBMISSION_NAME} folder.")

generate_submission_sasha(model_sasha_1)

Inference: 100%|██████████| 2000/2000 [00:30<00:00, 65.52it/s]

Inference complete. Zip the submission_blend_v2 folder.


In [23]:
from torchvision.models import swin_t, Swin_T_Weights


class MultiViewBEVModelSwin(nn.Module):
    def __init__(self, bev_h=48, bev_w=32, num_classes=1, num_cameras=4):
        super().__init__()
        
        # --- Backbone ---
        # resnet = resnet50(weights='DEFAULT')
        # self.encoder = nn.Sequential(*list(resnet.children())[:-2])
        swin = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
        self.encoder = nn.Sequential(*list(swin.children())[:-3])

        # for p in self.encoder.parameters():
        #     p.requires_grad = False
        
        self.neck = nn.Conv2d(768, 512, kernel_size=1)

        # --- BEV queries ---
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.num_classes = num_classes
        self.num_cameras = num_cameras
        
        self.bev_queries = nn.Parameter(torch.empty(bev_h * bev_w, 512))
        nn.init.xavier_normal_(self.bev_queries)
        
        # --- Transformer ---
        self.transformer = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(512, 8, batch_first=True),
            num_layers=4,
        )
        
        # --- Positional encoding (learnable) ---
        # self.pos_encoding = nn.Parameter(torch.empty(1, num_cameras * 8 * 16, 512))  # H_feat=8, W_feat=16
        # nn.init.xavier_normal_(self.pos_encoding)

        self.register_buffer("pos_embed", positionalencoding2d(8, 16, 512))  # H_feat=8, W_feat=16
        self.register_buffer("bev_pos_embed", positionalencoding2d(self.bev_h, self.bev_w, 512) )

        self.upsample = nn.Sequential(
            # Stage 1: coarse → medium (~48x32)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(size=(96,64), mode='bilinear', align_corners=False),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(),

            # Stage 2: medium → finer (~94x63)
            nn.BatchNorm2d(256),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(size=(188,126), mode='bilinear', align_corners=False),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(),

            # Stage 3: finer → final (188x126)
            nn.BatchNorm2d(128),
            # nn.Upsample(size=(188,126), mode='bilinear', align_corners=False),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            # Final projection
            nn.BatchNorm2d(64),
            nn.Conv2d(64, num_classes, kernel_size=1)
        )

        self.intrinsics_encoder = nn.Sequential(
            nn.BatchNorm1d(12),
            nn.Linear(12, 512),
            nn.ReLU(),
            nn.Linear(512, 512)
        )

        self.car2cams_encoder = nn.Sequential(
            nn.BatchNorm1d(16),
            nn.Linear(16, 512),
            nn.ReLU(),
            nn.Linear(512, 512)
        )


    def forward(self, x, intrinsics, car2cams):
        B, N, C, H, W = x.shape
        assert N == self.num_cameras, "Number of cameras mismatch"
        
        intrinsics = self.intrinsics_encoder(intrinsics.reshape(B * N, -1).float()).reshape(B, N, -1)
        car2cams = self.car2cams_encoder(car2cams.reshape(B * N, -1).float()).reshape(B, N, -1)

        # --- Extract features from each camera ---
        x = x.view(B * N, C, H, W)
        features = self.encoder(x)  # (B*N, 512, 8,16)
        features = self.neck(features)
        _, C_feat, H_feat, W_feat = features.shape
        features = features.view(B, N, C_feat, H_feat, W_feat)
        
        # --- Flatten camera + spatial dims for transformer ---
        features = features.permute(0, 1, 3, 4, 2)  # (B, N, H_feat, W_feat, C)
        features = features + self.pos_embed[None, None]  # (B, N, H_feat, W_feat, C)
        features = features.reshape(B, N, H_feat*W_feat, C_feat)  # (B, N, H_feat*W_feat, 512)
        features = features + intrinsics.unsqueeze(2) + car2cams.unsqueeze(2) 
        features = features.reshape(B, N*H_feat*W_feat, C_feat)  # (B, seq_len, 512)

        # --- Add positional encoding ---
        # features = features + self.pos_encoding  # (B, seq_len, 512)
        
        bev_tokens = self.bev_queries.reshape(self.bev_h, self.bev_w, C_feat) + self.bev_pos_embed
        bev_tokens = bev_tokens.view(self.bev_h*self.bev_w, C_feat)
        # --- Transformer decoder ---
        bev_tokens = self.transformer(
            self.bev_queries.unsqueeze(0).repeat(B,1,1),  # (B, bev_h*bev_w, 512)
            features
        )  # (B, bev_h*bev_w, 512)
        
        # --- Reshape to coarse BEV map ---
        bev_tokens = bev_tokens.transpose(1, 2).view(B, 512, self.bev_h, self.bev_w)  # (B, 512, H_bev, W_bev)
        
        # --- Conv upsampling to final BEV ---
        bev_logits = self.upsample(bev_tokens)  # (B, num_classes, 188, 126)
        
        return bev_logits


In [ ]:
model_sasha_2 = MultiViewBEVModelSwin()
model_sasha_2.load_state_dict(torch.load('./checkpoints/26_0.5714114289283753.pth'))
model_sasha_2 = model_sasha_2.to(CONFIG['device'])

In [25]:
generate_submission_sasha(model_sasha_2)

Inference: 100%|██████████| 2000/2000 [00:38<00:00, 51.61it/s]

Inference complete. Zip the submission_blend_v2 folder.


In [32]:
for filename in os.listdir(f'{SUBMISSION_NAME}/predicted_static_grids'):
    probs = np.load(f'{SUBMISSION_NAME}/predicted_static_grids/{filename}')
    pred = (probs / 3 > 0.5).astype('int') 
    np.save(f'{SUBMISSION_NAME}/predicted_static_grids/{filename}', pred)

In [33]:
import shutil

# Zip entire folder
shutil.make_archive(SUBMISSION_NAME, 'zip', SUBMISSION_NAME)


'/app/submission_blend_v2.zip'